# Data Cleaning

**Purpose:** Combine the three booking files, anonymize customer information, address data quality issues, extract useful features, and save a cleaned dataset.

**Input:** `2024.csv`, `2025.csv`, `2026.csv`

**Output:** `booking_clean.csv`

In [170]:
import sys
sys.path.append("..")

import pandas as pd
from pathlib import Path
from src.anonymize import anonymize 
from src.validate import validate

RAW = Path("../data/raw/")
OUT = Path("../data/processed")

FILES = ["2024.csv", "2025.csv", "2026.csv"]

log = []

## 1. Load & Combine

In [171]:
# Check if the file exits
for f in FILES:
    print(f, (RAW/f).exists())


2024.csv True
2025.csv True
2026.csv True


In [172]:
# Load data & combine
data = {}
for f in FILES:
    data[f] = pd.read_csv(RAW/f)
    print(f, data[f].shape)


# Check if the columns match
base = set(data["2024.csv"].columns)
for f in FILES:
    print(f, "matches 2024:", set(data[f].columns) == base)

# Stack the files together + create source column
frames = [] # create a list to hold the dataframes temporarily before combining
for f in FILES:
    df = data[f].copy()
    df["source"] = f
    frames.append(df)

df = pd.concat(frames, ignore_index=True)
print(df.shape)
log.append(("Combined 3 files", None, len(df)))
for col in df.columns:
    print(col)


2024.csv (6690, 11)
2025.csv (21720, 11)
2026.csv (21053, 11)
2024.csv matches 2024: True
2025.csv matches 2024: True
2026.csv matches 2024: True
(49463, 12)
date
created_by
number
phone_number
status
client_name
service_name
staff_name
service_price
start_time
end_time
source


## 2. Anonymize
The raw datefiles contain the name and phone number of customers, which should be anonymized with a new identifier.

In [173]:
# Rename to match the anynomization function
df = df.rename(columns={
    "phone_number": "phone",
    "client_name": "name"
})

df = anonymize(df, id_source="phone", drop_cols = ("name","phone"))

print("Unique customers:", df["id"].nunique())
print("Rows without customer_id:", df["id"].isna().sum())
print("name/phone still present:", {"name", "phone"} & set(df.columns))
log.append(("Anonymized customers", None, len(df)))



Unique customers: 5219
Rows without customer_id: 0
name/phone still present: set()


## 3. Clean

In [174]:
# Profile dataset
# Profile dataset
profile = pd.DataFrame({
    "non_null": df.count(),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
    "dtype": df.dtypes
})
profile = profile.sort_values("missing", ascending=False)
profile
df.describe(include="all")

,date,created_by,number,status,service_name,staff_name,service_price,start_time,end_time,source,id
count,49463,49463,4.946300e+04,49463,49462,49462,49462.000000,49462,49462,49463,49463
unique,801,2,NaN,3,57,25,NaN,50,50,3,5219
top,2025-12-23,Business,NaN,Completed,BIO GEL REFILL,Anna,NaN,09:30,14:00,2025.csv,d347a58376
freq,135,25761,NaN,44533,10939,7104,NaN,2554,1863,21720,3410
mean,NaN,NaN,4.997212e+06,NaN,NaN,NaN,35.487688,NaN,NaN,NaN,NaN
std,NaN,NaN,2.830138e+06,NaN,NaN,NaN,14.478360,NaN,NaN,NaN,NaN
min,NaN,NaN,9.575080e+05,NaN,NaN,NaN,3.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,2.415554e+06,NaN,NaN,NaN,27.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,4.639422e+06,NaN,NaN,NaN,38.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,7.399324e+06,NaN,NaN,NaN,45.000000,NaN,NaN,NaN,NaN


In [175]:
# Missing values
print("Missing values by column:")
print(df.isna().sum())
print("Rows with missing values:", df.isna().any(axis=1).sum())
before = len(df)
df = df.dropna()
print(f"Dropped {before - len(df)} rows; {len(df)} remain")
log.append(("Dropped rows with missing service/time/price", before - len(df), len(df)))


Missing values by column:
date             0
created_by       0
number           0
status           0
service_name     1
staff_name       1
service_price    1
start_time       1
end_time         1
source           0
id               0
dtype: int64
Rows with missing values: 1
Dropped 1 rows; 49462 remain


In [176]:
# Checking duplicates
print("Rows:", len(df))
print("Exact duplicate rows:", df.duplicated().sum())
df_noid = df.drop(columns="id")
print("Duplicates (without id):",df_noid.duplicated().sum())
log.append(("Checked duplicates", None, len(df)))

Rows: 49462
Exact duplicate rows: 0
Duplicates (without id): 0


In [177]:
# Data types parsing
    # Check datetime columns
print(df[["date", "start_time", "end_time"]].sample(5, random_state=1))

df["start"] = pd.to_datetime(df["date"] + " " + df["start_time"], errors="coerce")
df["end"]   = pd.to_datetime(df["date"] + " " + df["end_time"],   errors="coerce")
df["date_parsed"] = pd.to_datetime(df["date"], errors="coerce")

    # Numeric columns
df['service_price'] = pd.to_numeric(df['service_price'], errors='coerce')
log.append(("Parsed date and numeric columns", None, len(df)))


             date start_time end_time
44387  2026-03-31      12:45    13:30
25736  2025-02-25      15:15    16:15
12104  2025-10-01      10:30    11:00
8676   2025-12-03      17:00    18:15
19569  2025-06-14      12:45    13:00


In [178]:
# Fix HTML codes in service names (&amp; should be &)
before = df["service_name"].str.contains("&amp;", na=False).sum()

df["service_name"] = (df["service_name"]
                      .str.replace("&amp;", "&", regex=False)
                      .str.strip())

after = df["service_name"].str.contains("&amp;", na=False).sum()
print(f"Rows with &amp; before: {before}, after: {after}")
print("Unique services:", df["service_name"].nunique())

assert after == 0, "&amp; still present"
if before > 0:
    log.append(("Fixed HTML codes in service_name", before, len(df)))

Rows with &amp; before: 2604, after: 0
Unique services: 53


## 4. Feature Engineering

In [179]:
# Year, month, day of the week, hour, is_weekend
df["year"] = df["date_parsed"].dt.year
df["month"] = df["date_parsed"].dt.month
df["day_of_week"] = df["date_parsed"].dt.day_name()
df["hour"] = df["start"].dt.hour
df['is_weekend'] = df["day_of_week"].isin(["Saturday", "Sunday"])

# Duration in minutes
df['duration'] = (df['end'] - df['start']).dt.total_seconds() / 60

# Days since last booking
df = df.sort_values(by=["id", "start"])
df['days_since_last_booking'] = df.groupby('id')['start'].diff().dt.days

# Type of service
print(df["service_name"].value_counts(dropna=False))
SERVICE_GROUPS = {
    "Add-ons": [
        "NAIL ART", "AIR BRUSH DESIGN", "MEDIUM OR LONG LENGTH",
        "CAT EYES", "CHROME", "SHAPE CHANGE",
    ],
    "Natural Nails": [
        "MANICURE", "PEDICURE", "SHELLAC MANICURE", "SHELLAC PEDICURE",
        "MANI & PEDI", "SHELLAC MANI & PEDI",
        "SHELLAC POLISH CHANGE (HANDS)", "SHELLAC POLISH CHANGE (TOES)",
        "DELUXE PEDICURE (REGULAR)", "DELUXE PEDICURE (SHELLAC)",
        "DELUXE PEDICURE + MANICURE SHELLAC",
    ],
    "Kids Services": [
        "KID HAND POLISH", "KID TOE POLISH", "KID MANICURE",
        "KID PEDICURE", "KID MANICURE & PEDICURE",
    ],
    "Bare Necessities": [
        "HAND POLISH", "TOE POLISH", "NAIL REPAIR", "CUT DOWN",
        "ACRYLIC TAKE OFF", "SHELLAC TAKE OFF",
        "HAND POLISH WITH FRENCH", "TOE POLISH WITH FRENCH",
    ],
    "Waxing Services": [
        "EYEBROWS WAX", "UPPER LIP WAX", "CHIN WAX", "CHIN & SIDEBURNS WAX",
        "FULL FACE WAX", "UNDER ARMS WAX", "HALF ARMS WAX", "FULL ARMS WAX",
        "HAND OR FEET WAX", "TUMMY WAX", "CHEST WAX", "BACK WAX",
        "BIKINI LINE WAX", "BRAZILIAN WAX", "HALF LEG WAX", "FULL LEG WAX",
        "TINTING & EYEBROW WAX",
    ],
    "Nails Enhancement": [
        "BIO GEL REFILL", "BIO GEL NEW SET", "ACRYLIC REFILL", "ACRYLIC NEW SET",
        "UV GEL REFILL", "UV GEL NEW SET", "SOLAR REFILL", "SOLAR NEW SET",
    ],
}
# Flip to {service: group}. Uppercase the keys so capitalization can't cause a miss.
SERVICE_TO_TYPE = {s.upper(): g for g, items in SERVICE_GROUPS.items() for s in items}

df["service_type"] = df["service_name"].str.strip().str.upper().map(SERVICE_TO_TYPE)
unmapped = df.loc[df["service_type"].isna(), "service_name"].value_counts()
print("Unmapped:", len(unmapped), "services,", unmapped.sum(), "rows")
print(unmapped)

assert df["service_type"].notna().all(), "Some services are unmapped"
print(df["service_type"].value_counts())
log.append(("Feature Engineering", None, len(df)))


service_name
BIO GEL REFILL                        10939
PEDICURE                               6608
SHELLAC PEDICURE                       5198
ACRYLIC REFILL                         4570
SHELLAC MANICURE                       4249
ACRYLIC NEW SET                        3177
NAIL ART                               1800
SHELLAC MANI & PEDI                    1662
SHELLAC POLISH CHANGE (HANDS)          1207
BIO GEL NEW SET                        1181
UV GEL REFILL                          1036
MANI & PEDI                             928
MANICURE                                800
ACRYLIC TAKE OFF                        725
EYEBROWS WAX                            712
SHELLAC POLISH CHANGE (TOES)            536
MEDIUM OR LONG LENGTH                   529
NAIL REPAIR                             517
SOLAR REFILL                            389
SHELLAC TAKE OFF                        361
UV GEL NEW SET                          242
Deluxe Pedicure (Regular)               241
Deluxe Pedicure (Sh

## 5. Validation & Save

In [180]:
df = validate(df)

All checks passed: 49,462 rows, 22 columns


In [181]:
# Prepare and Save data
raw_cols = ["start_time", "end_time"]
df = df.drop(columns=raw_cols)
df = df.sort_values("date_parsed").reset_index(drop=True)

df.to_parquet(OUT / "booking_clean.parquet", index=False)
df.to_csv(OUT / "booking_clean.csv", index=False)

print(f"Saved {len(df):,} rows, {df.shape[1]} columns")

Saved 49,462 rows, 20 columns
